# Flex `use_channels` redesign — hardware smoke test

Drives the **new unified surface** on a real Flex: PLR-native `plate.column(c)` column
ops, single-tip `pick_up_tips(spot, use_channels=[7])` (the H1 front nozzle), and a
front-anchored **partial** column.

**Structure:** run the **setup** cells top-to-bottom once (deck → connect → home → trash).
After that, the three **pipetting** cells (full column, single H1 tip, front-4 partial) are
each self-contained — they pick their own tips, pipette, discard, and return the gantry to
home — so you can run them in **any order**. Each uses a different tip-rack column, so they
don't collide.

**This moves the robot.** Tip rack in **D1**, plate in **B1**, rows **A and C empty** (see
the deck rule below). Keep the e-stop in reach. Put liquid in the plate for real draws —
otherwise it aspirates air, fine for a motion check.

## Deck-layout rule for single / partial tip pickup

In a single- or partial-nozzle config the idle nozzles trail off one end of the head, so
keep the adjacent rows clear:

| pickup nozzle | idle nozzles trail toward | labware allowed in | keep EMPTY |
|---|---|---|---|
| **H1** (front) | rear | rows **B** and **D** | rows **A** and **C** |
| **A1** (rear)  | front | rows **A** and **C** | rows **B** and **D** |

This notebook uses the **H1** convention: tip rack in **D1**, plate in **B1**, rows A and C
empty.

## Setup — run these once, top to bottom

Set `FLEX_HOST` below to your Flex's IP or USB address (the same address the Opentrons App uses; port 31950). On a link-local connection (`169.254.x.x`) you may need to wake the interface first, e.g.:

```bash
ping -c2 -b <interface> 169.254.255.255   # wake the link, then plain HTTP routes
```

In [ ]:
FLEX_HOST = "169.254.1.1"  # <-- SET to your Flex's IP / USB address

from pylabrobot.opentrons import FlexHead8, OpentronsFlex
from pylabrobot.resources.opentrons import (
    FlexDeck,
    corning_96_wellplate_360ul_flat,
    flex_96_tiprack_50ul,
)

In [ ]:
deck = FlexDeck()
tip_rack = flex_96_tiprack_50ul(name="tips_01")
plate = corning_96_wellplate_360ul_flat(name="plate_01")
# H1 convention: labware only in rows B and D; rows A and C empty (see the deck rule).
deck.assign_child_at_slot(tip_rack, "D1")
deck.assign_child_at_slot(plate, "B1")
# The trash is a deck feature; grab its handle now that the slots are populated.
trash = deck.get_trash_area()

In [ ]:
flex = OpentronsFlex(deck, host=FLEX_HOST)
await flex.setup()

print("api_version:", flex.api_version)
print("robot_model:", flex.robot_model)
head = flex.left or flex.right
assert isinstance(head, FlexHead8), f"expected FlexHead8, got {type(head)}"
print("head:", head, "| mounted tips:", head.get_mounted_tips())

In [ ]:
# Finish setup: home the gantry. After this, the pipetting cells below are
# self-contained and can be run in any order.
await flex.home()

## Pipetting — run these in any order

Each cell picks its own tips (a different column each), pipettes, discards to the trash,
and homes. Re-running a cell tries to pick the same tips again, so run each once per pass.

### Full column — `plate.column(c)` (ALL layout)

In [ ]:
# PLR-native: pass the list of wells, not (rack, column).
await head.pick_up_tips(tip_rack.column(0))
print("mounted:", sum(1 for t in head.get_mounted_tips() if t is not None), "tips")
await head.aspirate(plate.column(0), volume=50)
await head.dispense(plate.column(0), volume=50)
await head.discard_tips(trash)
await flex.home()  # park at home after the trash drop
print("full column done; mounted:", sum(1 for t in head.get_mounted_tips() if t is not None))

### Single H1 tip — `use_channels=[7]` (SINGLE layout)

`use_channels=[7]` configures the **H1** front nozzle; the single-nozzle liquid op is
guarded by the same idle-nozzle clearance check as pickup. Plate is in **B1** with the row
behind it (**A1**) empty, so H1 clears — the aspirate/dispense is allowed.

In [ ]:
# Picks from column 1 (well A2); lands the tip on channel 7.
await head.pick_up_tips(tip_rack.get_item("A2"), use_channels=[7])
await head.aspirate(plate.get_item("A1"), volume=20)
await head.dispense(plate.get_item("A1"), volume=20)
await head.drop_single_tip(trash)
await flex.home()  # park at home after the trash drop
print("single done; mounted:", sum(1 for t in head.get_mounted_tips() if t is not None))

### Partial column — `use_channels=[4,5,6,7]` (QUADRANT layout)

Front nozzles **E,F,G,H** (channels 4-7) pick the **first 4 rows (A-D)** of column 4. This emits Opentrons' own QUADRANT config (`primaryNozzle = frontRightNozzle = "H1"`, `backLeftNozzle = "E1"`), anchored at the D-row tip.

**Why the first 4 and not the last 4:** all 8 nozzles descend to the same height, so the 4 *inactive* nozzles must be over empty space. Picking the first 4 rows puts the inactive rear nozzles **off the back of the rack** (behind row A); picking `[4:8]` (rows E-H) would put them over the A-D tips still in the column — a crash. The idle nozzles also trail into the empty **C1** row (rack in D1), per the deck rule.

**Bench-first:** this exact partial-column config has not been run on hardware — watch it, e-stop ready.

In [ ]:
# Front nozzles E,F,G,H (channels 4-7) pick the FIRST 4 rows (A-D) of column 4.
# Picking the first 4 keeps the inactive rear nozzles OFF the back of the rack (behind
# row A), clear of the E-H tips still in that column. (Picking the LAST 4 [4:8] would put
# the inactive nozzles over the A-D tips -> collision.)
await head.pick_up_tips(tip_rack.column(3)[0:4], use_channels=[4, 5, 6, 7])
print("partial channels with tips:",
      [i for i, t in enumerate(head.get_mounted_tips()) if t is not None])
await head.discard_tips(trash)
await flex.home()  # park at home after the trash drop
print("partial done; mounted:", sum(1 for t in head.get_mounted_tips() if t is not None))

## Teardown

In [ ]:
await flex.stop()